In [ ]:
import pandas as pd 
import sys 
sys.path.append("/workspace/")
from src.metrics import ClinicalQAEvaluator
from glob import glob  

result_root_dir = "/workspace/kor_med_opendataset/results/snuh_ClinicalQA_benchmark"

In [ ]:
from IPython.display import display
import os 
parquet_files = glob(f"{result_root_dir}/**/*.parquet", recursive=False)

all_summary_df = pd.DataFrame()
for parquet_file in parquet_files:
    model_name = parquet_file.split('/')[-2]
    evaluator = ClinicalQAEvaluator(parquet_file)

    # 성능 요약
    summary_df = evaluator.summary()
    
    # 맨 앞에 모델명 붙히기
    summary_df.insert(0, 'model', model_name)
    
    all_summary_df = pd.concat([all_summary_df, summary_df])
    
all_summary_df = all_summary_df.sort_values("accuracy (%)", ascending=True)
# all_summary_df.to_csv(f"{result_root_dir}/all_summary.csv", index=False)


In [ ]:
# 모델명을 '_'로 분리해 그룹(시리즈)와 이름을 분리해서, 시리즈(group)별로 accuracy 오름차순 정렬해서 보여주기
df_temp = all_summary_df.copy()
df_temp['model_group'] = df_temp['model'].apply(lambda x: x.split('_')[0])
df_temp['model_name'] = df_temp['model'].apply(lambda x: '_'.join(x.split('_')[1:]))

df_temp['mean_flops (GFlops)'] = (df_temp['mean_flops'] / 1e9).round(3)
df_temp['accuracy (%)'] = df_temp['accuracy (%)'].round(3)
df_temp['avg_time_per_token (s)'] = df_temp['avg_time_per_token (s)'].round(3)

display_cols = ['model_group', 'model_name', 'accuracy (%)', 'avg_time_per_token (s)', 'mean_flops (GFlops)']
# model_group별로 accuracy 오름차순 정렬
df_temp = df_temp.sort_values(['model_group', 'accuracy (%)'], ascending=[True, True])
display(df_temp[display_cols])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. 텍스트 겹침 방지 라이브러리 확인 (없으면 기본 텍스트 사용)
# ---------------------------------------------------------
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False
    print("Tip: 'pip install adjustText'를 하면 라벨이 겹치지 않게 자동 정렬됩니다.")

# ---------------------------------------------------------
# 2. 논문용 고품질 플롯 함수 정의
# ---------------------------------------------------------
def plot_medical_llm_benchmark(
    df,
    x_col='mean_flops',
    y_col='accuracy (%)',
    model_col='model',
    figsize=(12, 8),  # 논문용으로 시원한 크기
    dpi=300,
    title="Medical Domain Evaluation of Open-Source Small Large Language Models",
    save_path="medical_llm_eval.png"
):
    # 데이터 전처리: 그룹명과 모델 짧은 이름 추출
    plot_df = df.copy()
    
    # -------------------------------------------------------
    # [수정됨] FLOPs 단위를 GFLOPs로 변환 (1 GFLOPs = 1e9 FLOPs)
    # -------------------------------------------------------
    plot_df[x_col] = plot_df[x_col] / 1e9
    
    # 예: 'Llama-2_7B' -> Group: 'Llama-2', Short: '7B'
    plot_df['Group'] = plot_df[model_col].apply(lambda x: x.split('_')[0])
    plot_df['Short Name'] = plot_df[model_col].apply(lambda x: '_'.join(x.split('_')[1:]))

    # Seaborn 스타일 설정 (논문 스타일: 깔끔한 흰색 배경, 틱 강조)
    sns.set_context("paper", font_scale=1.5)
    sns.set_style("ticks", {'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.5})

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # -------------------------------------------------------
    # A. Pareto Frontier (효율성 경계선) 그리기
    # -------------------------------------------------------
    # X축(연산량) 정렬 후, 현재까지의 최대 정확도를 가진 점들을 연결
    sorted_df = plot_df.sort_values(by=x_col)
    pareto_points = []
    current_max_y = -np.inf
    
    for _, row in sorted_df.iterrows():
        if row[y_col] > current_max_y:
            pareto_points.append((row[x_col], row[y_col]))
            current_max_y = row[y_col]
            
    if pareto_points:
        px, py = zip(*pareto_points)
        ax.plot(px, py, color='gray', linestyle='--', linewidth=1.5, alpha=0.6, zorder=1, label='Pareto Frontier (Efficiency)')

    # -------------------------------------------------------
    # B. 메인 산점도 (Scatter Plot)
    # -------------------------------------------------------
    # 마커 모양 리스트 (그룹별로 모양을 다르게 하여 흑백 인쇄 시에도 구분 가능)
    markers = ['o', 'D', 's', '^', 'v', 'X', 'P', '*', 'h', '<']
    unique_groups = plot_df['Group'].unique()
    marker_map = {g: markers[i % len(markers)] for i, g in enumerate(unique_groups)}

    sns.scatterplot(
        data=plot_df,
        x=x_col,
        y=y_col,
        hue='Group',
        style='Group',
        markers=marker_map,
        palette='tab10',  # 명확한 색상 구분
        s=250,            # 마커 크기 확대
        alpha=0.9,
        edgecolor='white',
        linewidth=1.5,
        ax=ax,
        zorder=3
    )

    # -------------------------------------------------------
    # C. 축 및 타이틀 설정
    # -------------------------------------------------------
    ax.set_xscale('log')
    # [수정됨] X축 라벨 변경 (GFLOPs)
    ax.set_xlabel("Computational Cost (GFLOPs, Log Scale)", fontweight='bold', labelpad=12)
    ax.set_ylabel("Accuracy (%)", fontweight='bold', labelpad=12)
    
    if title:
        ax.set_title(title, fontweight='bold', fontsize=18, pad=20)

    # 테두리 정리 (오른쪽, 위쪽 테두리 제거 - Tufte 스타일)
    sns.despine(trim=True, offset=10)

    # -------------------------------------------------------
    # D. 라벨링 (adjustText 사용 권장)
    # -------------------------------------------------------
    texts = []
    for _, row in plot_df.iterrows():
        texts.append(
            ax.text(
                row[x_col], 
                row[y_col], 
                row['Short Name'], 
                fontsize=11, 
                color='#333333',
                fontweight='medium'
            )
        )

    if HAS_ADJUST_TEXT:
        # 화살표 색상을 회색으로 은은하게 처리
        adjust_text(
            texts, 
            ax=ax,
            arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=0.5),
            expand_points=(1.2, 1.2),
            force_text=(0.3, 0.5)
        )

    # -------------------------------------------------------
    # E. 범례 (Legend) 최적화
    # -------------------------------------------------------
    # 범례를 그래프 안쪽(우측 하단)에 배치하되, 반투명 박스로 처리
    ax.legend(
        title="Model Family",
        loc='lower right',
        bbox_to_anchor=(1.0, 0.05),
        frameon=True,
        fancybox=True,
        framealpha=0.9,
        edgecolor='0.8',
        fontsize=11,
        title_fontsize=12
    )

    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Figure saved successfully to: {save_path}")

    plt.show()

plot_medical_llm_benchmark(all_summary_df)

In [ ]:
import pandas as pd 


df = pd.read_parquet("/workspace/kor_med_opendataset/results/snuh_ClinicalQA_benchmark/Qwen_Qwen3-8B/Qwen_Qwen3-8B_detailed.parquet")
df

In [ ]:
all_summary_df

In [ ]:
from IPython.display import display
import json
js_files = glob(f"{result_root_dir}/**/*.json", recursive=False)

js_dict = {}
for js_file in js_files:
    model_name = js_file.split('/')[-2]
    with open(js_file, 'r') as f:
        js_dict[model_name] = json.load(f)
import pandas as pd

def seconds_to_hms(seconds):
    """Convert seconds to HH:MM:SS format"""
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    sec = seconds % 60
    return f"{hours:02}:{minutes:02}:{sec:02}"

# Transform dict into list of dicts for DataFrame construction
records = []
for key, v in js_dict.items():
    rec = dict(model_group=key)
    rec.update(v)
    rec['total_time_hms'] = seconds_to_hms(v['total_time_s'])
    records.append(rec)

df = pd.DataFrame(records)

# Move total_time_hms next to total_time_s
if 'total_time_s' in df.columns and 'total_time_hms' in df.columns:
    cols = list(df.columns)
    time_idx = cols.index('total_time_s')
    cols.insert(time_idx + 1, cols.pop(cols.index('total_time_hms')))
    df = df[cols]

# Display DataFrame as usual
display(df)

# Sum total_time_s and show as HH:MM:SS
if 'total_time_s' in df.columns:
    total_sec = df['total_time_s'].sum()
    total_hms = seconds_to_hms(total_sec)
    print(f"\n전체 모델 total_time_s 합계: {total_sec:.2f}초 → {total_hms} (HH:MM:SS)")